# Harden `dla_engine_top` on GF180 with LibreLane — 3.3 V single-supply (Google Colab)

Runs the **11-macro variant** (A+B+C buffers all SRAM) using the **3.3 V**
standard-cell library `gf180mcu_as_sc_mcu7t3v3` instead of the foundry 5 V
cells. The logic then shares the OCD SRAM's 3.3 V rail → **single supply, no
level shifting**. Run the cells top to bottom.

**Before you start:** on your PC, zip these folders from `APIC_New/` into `apic_a.zip`:
- `rtl/`
- `gf180mcu_ocd_ip_sram__sram256x8m8wm1/`
- `librelane/`
- `3V3lib/`  ← **NEW** — contains the 3.3 V std-cell library `gf180mcu_as_sc_mcu7t3v3-main/`

The layout on Colab must end up as
`/content/APIC_A/{rtl, gf180mcu_ocd_ip_sram__sram256x8m8wm1, librelane, 3V3lib}`
so the `dir::../rtl/...` paths in `config.yaml` resolve and Step 4 can find the lib.

**How the 3.3 V library gets in:** the gf180mcuD PDK that LibreLane downloads
ships only the 5 V cells, so we (Step 3) download the PDK, (Step 4) copy the
3.3 V lib into it, then (Step 5) run with `--manual-pdk` so LibreLane uses our
patched PDK as-is.

_Runtime: ~15–25 min setup (Nix + PDK download), then ~30–60 min for the 11-macro flow._

## Step 1 — upload and unpack your project

In [ ]:
from google.colab import files
import os, glob, shutil, subprocess

print('Pick apic_a.zip ...')
up = files.upload()                 # choose apic_a.zip
zip_name = next(iter(up))

os.makedirs('/content/APIC_A', exist_ok=True)
get_ipython().system('unzip -q -o "{}" -d /content/APIC_A'.format(zip_name))

# If the zip wrapped everything in a top folder, flatten it so the project
# dirs sit directly under /content/APIC_A.
NEEDED = ('rtl', 'gf180mcu_ocd_ip_sram__sram256x8m8wm1', 'librelane', '3V3lib')
if not os.path.isdir('/content/APIC_A/librelane'):
    for cand in glob.glob('/content/APIC_A/*/librelane'):
        base = os.path.dirname(cand)
        for sub in NEEDED:
            src = os.path.join(base, sub)
            if os.path.isdir(src):
                shutil.move(src, '/content/APIC_A/{}'.format(sub))
        break

print('\nContents of /content/APIC_A:')
get_ipython().system('ls /content/APIC_A')
assert os.path.isfile('/content/APIC_A/librelane/config.yaml'), \
    'config.yaml not found - check your zip layout'

# Locate the 3.3V std-cell library (handles either zip layout).
as_pdk = subprocess.run(
    "find /content/APIC_A -type d -path '*gf180mcu_as_sc_mcu7t3v3-main/pdk' | head -1",
    shell=True, capture_output=True, text=True).stdout.strip()
assert as_pdk, \
    '3.3V std-cell lib (gf180mcu_as_sc_mcu7t3v3-main/pdk) not found - add 3V3lib/ to the zip'
print('\nOK: config.yaml found; 3.3V std-cell lib at', as_pdk)

## Step 2 — install Nix (the tool manager LibreLane uses)

Uses the **Determinate Systems** installer with `--init none` because Colab runs as root with **no systemd and no `nixbld` group** — the plain nixos.org installer fails there. Takes ~1–2 min.

In [ ]:
%%bash
# Clean up any half-finished install, then install Nix in a way that works
# on Colab (root, no systemd, no nixbld group).
sudo rm -rf /nix /etc/nix 2>/dev/null
curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix | \
  sh -s -- install linux --init none --no-confirm \
  --extra-conf "sandbox = false"
echo 'DONE installing nix (each new code cell is a fresh shell, so we re-source it before use)'

## Step 3 — download the gf180mcuD PDK

First LibreLane invocation. It pulls LibreLane + the EDA tools from the binary
cache (`--accept-flake-config`) and downloads the **gf180mcuD** PDK into
`/content/pdk`.

> ⚠️ **This run is EXPECTED to stop early** with an error that the std-cell
> library `gf180mcu_as_sc_mcu7t3v3` was not found. That is fine — all we need
> here is the PDK on disk; we install the 3.3 V lib into it in Step 4. The cell
> still finishes "green" and prints a download check at the end.

In [ ]:
%%bash
. /nix/var/nix/profiles/default/etc/profile.d/nix-daemon.sh
export NIX_REMOTE=
export PDK_ROOT=/content/pdk
mkdir -p "$PDK_ROOT"

# Download the PDK (ciel) by letting LibreLane resolve it. NO --manual-pdk here
# so ciel actually fetches it. Expected to abort at the std-cell-library load.
nix run --accept-flake-config --option sandbox false \
    github:librelane/librelane -- \
    /content/APIC_A/librelane/config.yaml --pdk-root "$PDK_ROOT" || true

echo
echo "=== PDK download check (expect gf180mcuD with the 5V lib present) ==="
ls "$PDK_ROOT" || true
if ls "$PDK_ROOT"/gf180mcuD/libs.ref/gf180mcu_fd_sc_mcu7t5v0 >/dev/null 2>&1; then
  echo "OK: gf180mcuD PDK downloaded to $PDK_ROOT/gf180mcuD"
else
  echo "WARNING: gf180mcuD not found yet - re-run this cell (download may have been interrupted)."
fi

## Step 4 — install the 3.3 V std-cell library into the PDK

Targeted copy of just the two directories the digital flow needs:
`libs.ref/gf180mcu_as_sc_mcu7t3v3` (lib/lef/gds/techlef/verilog) and
`libs.tech/librelane/gf180mcu_as_sc_mcu7t3v3` (the LibreLane integration:
9 STA corners, placement site, tie/fill/decap/diode/clkbuff/tap roles).

We deliberately do **not** copy the lib's `libs.tech/magic` or `libs.tech/xschem`
— those would overwrite core gf180mcuD tech files and aren't used by the
RTL→GDS flow.

In [ ]:
%%bash
export PDK_ROOT=/content/pdk
DST="$PDK_ROOT/gf180mcuD"
AS_PDK=$(find /content/APIC_A -type d -path '*gf180mcu_as_sc_mcu7t3v3-main/pdk' | head -1)
echo "Lib source : $AS_PDK"
echo "PDK target : $DST"

cp -r "$AS_PDK/libs.ref/gf180mcu_as_sc_mcu7t3v3"              "$DST/libs.ref/"
mkdir -p "$DST/libs.tech/librelane"
cp -r "$AS_PDK/libs.tech/librelane/gf180mcu_as_sc_mcu7t3v3"   "$DST/libs.tech/librelane/"

echo
echo "=== verify the 3.3V lib is now in the PDK ==="
echo "-- liberty corners (expect ff_n40C_3v60 / ss_125C_3v00 / tt_025C_3v30):"
ls "$DST/libs.ref/gf180mcu_as_sc_mcu7t3v3/lib/" | grep '\.lib$' || true
if [ -f "$DST/libs.tech/librelane/gf180mcu_as_sc_mcu7t3v3/config.tcl" ]; then
  echo "OK: libs.tech/librelane/gf180mcu_as_sc_mcu7t3v3/config.tcl present"
else
  echo "ERROR: config.tcl missing - the merge did not land where expected."
fi

## Step 5 — run the LibreLane flow (3.3 V)

`--manual-pdk` makes LibreLane use the PDK at `--pdk-root` **as-is** — so ciel
won't re-fetch and undo the Step-4 merge. The 3.3 V lib is now installed, so
`STD_CELL_LIBRARY: gf180mcu_as_sc_mcu7t3v3` (set in `config.yaml`) resolves and
the 9 corners come from its `config.tcl`.

If the run finishes you'll see `flow complete` near the end of the log.
This is the **first 3.3 V run**, so don't assume the old 5 V signoff carries
over (see Step 6).

In [ ]:
%%bash
. /nix/var/nix/profiles/default/etc/profile.d/nix-daemon.sh
export NIX_REMOTE=
export PDK_ROOT=/content/pdk

nix run --accept-flake-config --option sandbox false \
    github:librelane/librelane -- \
    /content/APIC_A/librelane/config.yaml \
    --pdk-root "$PDK_ROOT" --manual-pdk

## Step 6 — check signoff metrics and download the GDS

Expect `design__instance__count__class:macro = 11` and zero DRC/LVS errors.

> **First 3.3 V run — re-validate, don't trust the old numbers.** The floorplan
> (`DIE_AREA`, `PL_TARGET_DENSITY_PCT: 56`, `CLOCK_PERIOD: 75`) and the
> 0-antenna result were tuned for the 5 V cells. The 3.3 V cells have a
> different row height and timing, so `antenna__violating__nets`,
> `timing__setup__ws`, and `timing__hold__ws` may differ. Read them here and
> re-tune `config.yaml` if needed.

In [ ]:
import glob, csv, os

runs = sorted(glob.glob('/content/APIC_A/librelane/runs/*'))
assert runs, 'No runs/ directory - did Step 5 complete?'
run = runs[-1]
print('Run dir:', run)

metrics = os.path.join(run, 'final', 'metrics.csv')
want = {
    'design__instance__count__class:macro': 'SRAM macros (expect 11)',
    'design__instance__count':              'total instances',
    'magic__drc_error__count':              'Magic DRC errors (expect 0)',
    'klayout__drc_error__count':            'KLayout DRC errors',
    'design__lvs_error__count':             'Netgen LVS errors (expect 0)',
    'antenna__violating__nets':             'antenna violations',
    'timing__setup__ws':                    'setup worst slack ns (>=0)',
    'timing__hold__ws':                     'hold worst slack ns (>=0)',
}
if os.path.isfile(metrics):
    found = {}
    with open(metrics) as fh:
        for row in csv.reader(fh):
            if row and row[0] in want:
                found[row[0]] = row[1] if len(row) > 1 else ''
    for k, label in want.items():
        print('  {:32s} {}'.format(label, found.get(k, '(missing)')))
else:
    print('metrics.csv not found at', metrics)

In [ ]:
# Confirm the 11 SRAM macros are real black boxes in the synthesized netlist
# (not flip-flops). Expect 11 instance lines.
import glob
nl = glob.glob('{}/*-yosys-synthesis/dla_engine_top.nl.v'.format(run))
if nl:
    get_ipython().system('grep -n "gf180mcu_ocd_ip_sram__sram256x8m8wm1 " "{}"'.format(nl[0]))
    print('(expect 11 SRAM instance lines above)')
else:
    print('synthesis netlist not found (path layout may differ between LibreLane versions)')

In [ ]:
# Confirm the logic mapped to the 3.3V cells (sanity: should see the AS lib name).
import glob
nl = glob.glob('{}/*-yosys-synthesis/dla_engine_top.nl.v'.format(run))
if nl:
    get_ipython().system('grep -c "gf180mcu_as_sc_mcu7t3v3__" "{}"'.format(nl[0]))
    print('(non-zero = logic mapped to the 3.3V std cells, as intended)')
else:
    print('synthesis netlist not found')

In [ ]:
# Download the final GDS to view in KLayout on your PC.
import glob
from google.colab import files
gds = glob.glob('{}/final/gds/dla_engine_top.gds'.format(run))
if gds:
    print('Downloading', gds[0])
    files.download(gds[0])
else:
    print('No final GDS found - check the Step 5 log for errors.')